In [ ]:


from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    recall_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier

SEED = 13

# Versión reducida por tiempo de computación.

N_OUTER_SPLITS = 5
N_OUTER_REPEATS = 5


N_INNER_SPLITS = 5
N_INNER_REPEATS = 4

BASE = Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2")
PREPARED = BASE / "Prepared"
INPUT_CSV = PREPARED / "04_only_eeg.csv"

PARAM_GRID = {
    "xgb__max_depth": list(range(3, 12)),
    "xgb__learning_rate": [0.001, 0.01, 0.1, 1.0],
    "xgb__n_estimators": [50, 100],
}

OUT_DIR = PREPARED / "EEG_XGBoost_NarrativeWise"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reutiliza los mismos folds que el modelo conversation-level 
FOLDS_CSV = PREPARED / "EEG_XGBoost_Conversation" / "eeg_outer_folds_by_subject_repeated.csv"

In [ ]:

#  CARGAR DATOS EEG 


df = pd.read_csv(INPUT_CSV)

# Normalizar identificador de sujeto y etiqueta.
df["subject_id"] = df["subject_id"].astype(str).str.upper().str.strip()
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)


narratives = ["Neutral1", "Neutral2", "Happy", "Sad", "Angry", "Relax"]

df["narrative"] = df["avatar"].astype(str)
df = df[df["narrative"].isin(narratives)].copy()

# Features EEG: 27 variables = 9 canales × 3 bandas.
meta_cols = {
    "subject_id",
    "avatar",
    "narrative",
    "label",
    "phq",
    "outer_fold",
    "outer_repeat",
}

feature_cols = [
    c for c in df.columns
    if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])
]

# Comprobación de seguridad: no debería haber valores faltantes.
missing_total = int(df[feature_cols].isna().sum().sum())
print("Valores faltantes en features EEG:", missing_total)

if missing_total > 0:
    raise ValueError(
        "Hay valores faltantes en EEG. Revisa el CSV antes de entrenar."
    )

# Resumen rápido del dataset.
print("Filas EEG:", len(df))
print("Sujetos EEG:", df["subject_id"].nunique())
print("Variables EEG:", len(feature_cols))
print("Narrativas:", narratives)

print("\nClases por sujeto:")
display(df.drop_duplicates("subject_id")["label"].value_counts().sort_index())

print("\nConversaciones por sujeto:")
display(df.groupby("subject_id").size().value_counts().sort_index())

print("\nFilas por narrativa y clase:")
display(pd.crosstab(df["narrative"], df["label"]).reindex(narratives))

Valores faltantes en features EEG: 0
Filas EEG: 558
Sujetos EEG: 94
Variables EEG: 27
Narrativas: ['Neutral1', 'Neutral2', 'Happy', 'Sad', 'Angry', 'Relax']
Clases por sujeto:


label
0    55
1    39
Name: count, dtype: int64

Filas por narrativa:


label,0,1
narrative,,
Neutral1,55,39
Neutral2,54,38
Happy,55,39
Sad,55,39
Angry,54,38
Relax,54,38


In [ ]:
# FUNCIONES


def get_repeated_group_splits(X, y, groups, n_splits=N_INNER_SPLITS, n_repeats=N_INNER_REPEATS, seed=SEED):
    """Inner CV repetida, estratificada y agrupada por sujeto."""
    splits = []
    for rep in range(n_repeats):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed + rep)
        splits.extend(list(cv.split(X, y, groups)))
    return splits


def make_outer_folds(subjects):
    """Outer CV: 5 folds repetido 5 veces, estratificado por sujeto."""
    rows = []
    for rep in range(1, N_OUTER_REPEATS + 1):
        cv = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=SEED + rep)
        for fold, (_, test_idx) in enumerate(cv.split(subjects["subject_id"], subjects["label"]), start=1):
            tmp = subjects.iloc[test_idx].copy()
            tmp["outer_repeat"] = rep
            tmp["outer_fold"] = fold
            rows.append(tmp)
    return pd.concat(rows, ignore_index=True)


def make_sample_weight(y):
    """Pesos inversos a la frecuencia de clase dentro del train fold."""
    counts = y.value_counts()
    return y.map({cls: len(y) / (len(counts) * n) for cls, n in counts.items()}).values


def make_model():
    """Pipeline: escalado dentro del fold + XGBoost."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("xgb", XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            random_state=SEED,
            n_jobs=-1,
        )),
    ])


def safe_auc(y_true, y_prob):
    try:
        return roc_auc_score(y_true, y_prob)
    except ValueError:
        return np.nan


def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": safe_auc(y_true, y_prob),
        "recall_pos": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_neg": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


def summarize_metrics(metrics_df):
    cols = ["accuracy", "balanced_accuracy", "f1", "auc", "recall_pos", "recall_neg"]
    summary = pd.DataFrame({
        "mean": metrics_df[cols].mean(),
        "std": metrics_df[cols].std(),
    }).round(3)
    return summary

In [ ]:

# 3) CARGAR


subjects = df[["subject_id", "label"]].drop_duplicates("subject_id").reset_index(drop=True)

if FOLDS_CSV.exists():
    folds = pd.read_csv(FOLDS_CSV)
    folds["subject_id"] = folds["subject_id"].astype(str).str.upper().str.strip()
    print("Folds cargados desde:", FOLDS_CSV)
else:
    folds = make_outer_folds(subjects)
    print("Folds creados desde cero.")

folds.to_csv(OUT_DIR / "eeg_narrativewise_outer_folds_by_subject_repeated.csv", index=False)

print("Particiones externas:", folds[["outer_repeat", "outer_fold"]].drop_duplicates().shape[0])
display(pd.crosstab([folds["outer_repeat"], folds["outer_fold"]], folds["label"]).head(10))

Folds cargados desde: C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\EEG_XGBoost_Conversation_PaperLike_Repeat5_NoKNN\eeg_outer_folds_by_subject_repeated.csv
Particiones externas: 25


label                     0  1
outer_repeat outer_fold       
1            1           11  8
             2           11  8
             3           11  8
             4           11  8
             5           11  7
2            1           11  8
             2           11  8
             3           11  8
             4           11  8
             5           11  7

In [ ]:

# NESTED CV 


all_narrative_predictions = []
all_final_predictions = []
all_metrics = []
all_best_params = []

for (rep, fold), test_subjects in folds.groupby(["outer_repeat", "outer_fold"]):
    print(f"\n===== REPEAT {rep} | FOLD {fold} =====")
    test_ids = set(test_subjects["subject_id"])
    fold_predictions = []

    for narrative in narratives:
        train_n = df[(~df["subject_id"].isin(test_ids)) & (df["narrative"] == narrative)].copy()
        test_n = df[(df["subject_id"].isin(test_ids)) & (df["narrative"] == narrative)].copy()

        if train_n["label"].nunique() < 2 or test_n.empty:
            print(f"{narrative}: saltada")
            continue

        X_train = train_n[feature_cols]
        y_train = train_n["label"]
        groups_train = train_n["subject_id"]
        X_test = test_n[feature_cols]

        grid = GridSearchCV(
            estimator=make_model(),
            param_grid=PARAM_GRID,
            scoring="roc_auc",
            cv=get_repeated_group_splits(X_train, y_train, groups_train),
            n_jobs=-1,
            refit=True,
        )

        grid.fit(X_train, y_train, xgb__sample_weight=make_sample_weight(y_train))

        test_n = test_n.copy()
        test_n["y_prob"] = grid.best_estimator_.predict_proba(X_test)[:, 1]

        # Una probabilidad por sujeto y narrativa.
        pred_n = test_n.groupby(["subject_id", "label", "narrative"], as_index=False)["y_prob"].mean()
        pred_n["outer_repeat"] = rep
        pred_n["outer_fold"] = fold

        fold_predictions.append(pred_n)
        all_narrative_predictions.append(pred_n)

        all_best_params.append({
            "outer_repeat": rep,
            "outer_fold": fold,
            "narrative": narrative,
            "best_inner_auc": grid.best_score_,
            "best_max_depth": grid.best_params_["xgb__max_depth"],
            "best_learning_rate": grid.best_params_["xgb__learning_rate"],
            "best_n_estimators": grid.best_params_["xgb__n_estimators"],
        })

        print(f"{narrative}: best inner AUC={grid.best_score_:.3f}")

    # Predicción final por sujeto: media de probabilidades de las narrativas disponibles.
    pred_narratives = pd.concat(fold_predictions, ignore_index=True)
    pred_subject = pred_narratives.groupby(["subject_id", "label"], as_index=False)["y_prob"].mean()
    pred_subject["y_pred"] = (pred_subject["y_prob"] >= 0.5).astype(int)
    pred_subject["outer_repeat"] = rep
    pred_subject["outer_fold"] = fold

    m = compute_metrics(pred_subject["label"], pred_subject["y_pred"], pred_subject["y_prob"])
    m.update({"outer_repeat": rep, "outer_fold": fold})

    all_final_predictions.append(pred_subject)
    all_metrics.append(m)

    print("Best params:", grid.best_params_)
    print("Best AUC inner:", round(grid.best_score_, 3))

    print("Subject metrics:")
    print("  AUC:", round(m["auc"], 3))
    print("  Accuracy:", round(m["accuracy"], 3))
    print("  Balanced accuracy:", round(m["balanced_accuracy"], 3))
    print("  F1:", round(m["f1"], 3))
    print("  Recall DS:", round(m["recall_pos"], 3))
    print("  Recall HC:", round(m["recall_neg"], 3))

print("\nProceso terminado.")


===== REPEAT 1 | FOLD 1 =====
Neutral1: best inner AUC=0.786
Neutral2: best inner AUC=0.739
Happy: best inner AUC=0.713
Sad: best inner AUC=0.704
Angry: best inner AUC=0.698
Relax: best inner AUC=0.602
Best params: {'xgb__learning_rate': 1.0, 'xgb__max_depth': 3, 'xgb__n_estimators': 50}
Best AUC inner: 0.602
Subject metrics:
  AUC: 0.784
  Accuracy: 0.737
  Balanced accuracy: 0.739
  F1: 0.706
  Recall DS: 0.75
  Recall HC: 0.727

===== REPEAT 1 | FOLD 2 =====
Neutral1: best inner AUC=0.740
Neutral2: best inner AUC=0.730
Happy: best inner AUC=0.796
Sad: best inner AUC=0.765
Angry: best inner AUC=0.574
Relax: best inner AUC=0.664
Best params: {'xgb__learning_rate': 1.0, 'xgb__max_depth': 5, 'xgb__n_estimators': 100}
Best AUC inner: 0.664
Subject metrics:
  AUC: 0.739
  Accuracy: 0.632
  Balanced accuracy: 0.597
  F1: 0.462
  Recall DS: 0.375
  Recall HC: 0.818

===== REPEAT 1 | FOLD 3 =====
Neutral1: best inner AUC=0.751
Neutral2: best inner AUC=0.763
Happy: best inner AUC=0.673
Sad: 

In [ ]:
# GUARDAMOS RESULTADOS


narrative_predictions = pd.concat(all_narrative_predictions, ignore_index=True)
final_predictions = pd.concat(all_final_predictions, ignore_index=True)
metrics_df = pd.DataFrame(all_metrics)
best_params_df = pd.DataFrame(all_best_params)

narrative_predictions.to_csv(OUT_DIR / "eeg_narrativewise_probabilities_by_narrative.csv", index=False)
final_predictions.to_csv(OUT_DIR / "eeg_narrativewise_predictions_subject_level.csv", index=False)
metrics_df.to_csv(OUT_DIR / "eeg_narrativewise_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "eeg_narrativewise_best_params.csv", index=False)

print("Archivos guardados en:", OUT_DIR)
print("\nMétricas principales: media ± desviación por outer fold")
display(summarize_metrics(metrics_df))

Archivos guardados en: C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\EEG_XGBoost_NarrativeWise_PaperLike_Repeat5_NoKNN

Métricas principales: media ± desviación por outer fold


,mean,std
accuracy,0.723,0.099
balanced_accuracy,0.705,0.110
f1,0.631,0.160
auc,0.775,0.110
recall_pos,0.604,0.195
recall_neg,0.807,0.096
